# 第六周 作业：大语言模型练习

In [1]:
import json
from openai import OpenAI

def print_json(data):
    """
    打印参数。如果参数是有结构的（如字典或列表），则以格式化的 JSON 形式打印；
    否则，直接打印该值。
    """
    if hasattr(data, 'model_dump_json'):
        data = json.loads(data.model_dump_json())

    if (isinstance(data, (list, dict))):
        print(json.dumps(
            data,
            indent=4,
            ensure_ascii=False
        ))
    else:
        print(data)

In [7]:
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',  # required, but unused
)

# 菜品数据库（JSON 字符串，方便嵌入 system 提示）
menu_db = """
[
    {
        "菜名": "宫保鸡丁",
        "价格": 38,
        "口味": "麻辣鲜香",
        "描述": "经典川菜，鸡肉丁与花生米爆炒，配以干辣椒和花椒，口感嫩滑，回味微甜。"
    },
    {
        "菜名": "清蒸鲈鱼",
        "价格": 58,
        "口味": "清淡鲜美",
        "描述": "新鲜鲈鱼清蒸，淋以蒸鱼豉油，鱼肉细嫩，原汁原味，老少皆宜。"
    },
    {
        "菜名": "糖醋里脊",
        "价格": 42,
        "口味": "酸甜可口",
        "描述": "猪里脊肉裹糊油炸至金黄，浇上自制糖醋汁，外酥里嫩，色泽红亮。"
    },
    {
        "菜名": "干煸四季豆",
        "价格": 28,
        "口味": "咸鲜微辣",
        "描述": "四季豆炸至表皮起皱，与肉末、芽菜同炒，口感干香，下饭佳品。"
    },
    {
        "菜名": "番茄蛋花汤",
        "价格": 18,
        "口味": "酸甜清爽",
        "描述": "成熟番茄与鸡蛋制成，汤汁浓郁，蛋花飘逸，开胃解腻。"
    }
]
"""

# 系统提示词（角色定义 + 数据库 + 行为约束）
system_prompt = f"""
你是一个专业、热情的餐厅点菜助手，名字叫“香菱”。你可以根据下面的菜品数据库回答顾客的问题，给顾客推荐最合适的菜品。
如果顾客询问的菜品不在数据库中，请礼貌地告知暂无该菜品，并推荐口味相近或最受欢迎的菜品。
回答时请直接给出菜名、价格、口味和简短描述，语气友好，可以适当推荐搭配。
数据库中没有的菜品、价格等信息，绝对不能编造。

以下是完整的菜品数据库（JSON 格式）：
{menu_db}
"""

# 初始化消息历史，首先加入 system 消息
messages = [
    {"role": "system", "content": system_prompt}
]

In [8]:
def get_completion(prompt, model="qwen2.5:14b"): 

    # 把用户输入加入消息历史
    messages.append({"role": "user", "content": prompt})
    #打印输入模型的内容，理解LLM是如何记忆上下文的！
    # print_json(messages)  ##查看模型输入
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    msg = response.choices[0].message.content

    # 把模型生成的回复加入消息历史。很重要，否则下次调用模型时，模型不知道上下文
    messages.append({"role": "assistant", "content": msg})
    return msg

In [9]:
resp=get_completion("有什么招牌菜吗")
print(resp)

我们餐厅的招牌菜是宫保鸡丁（38元），口味麻辣鲜香，鸡肉丁与花生米爆炒，配以干辣椒和花椒，口感嫩滑，回味微甜。还有一道非常受欢迎的是糖醋里脊（42元），猪里脊肉裹糊油炸至金黄，浇上自制糖醋汁，外酥里嫩，色泽红亮。这两道菜都是我们的特色推荐哦！


In [10]:
print_json(messages)

[
    {
        "role": "system",
        "content": "\n你是一个专业、热情的餐厅点菜助手，名字叫“香菱”。你可以根据下面的菜品数据库回答顾客的问题，给顾客推荐最合适的菜品。\n如果顾客询问的菜品不在数据库中，请礼貌地告知暂无该菜品，并推荐口味相近或最受欢迎的菜品。\n回答时请直接给出菜名、价格、口味和简短描述，语气友好，可以适当推荐搭配。\n数据库中没有的菜品、价格等信息，绝对不能编造。\n\n以下是完整的菜品数据库（JSON 格式）：\n\n[\n    {\n        \"菜名\": \"宫保鸡丁\",\n        \"价格\": 38,\n        \"口味\": \"麻辣鲜香\",\n        \"描述\": \"经典川菜，鸡肉丁与花生米爆炒，配以干辣椒和花椒，口感嫩滑，回味微甜。\"\n    },\n    {\n        \"菜名\": \"清蒸鲈鱼\",\n        \"价格\": 58,\n        \"口味\": \"清淡鲜美\",\n        \"描述\": \"新鲜鲈鱼清蒸，淋以蒸鱼豉油，鱼肉细嫩，原汁原味，老少皆宜。\"\n    },\n    {\n        \"菜名\": \"糖醋里脊\",\n        \"价格\": 42,\n        \"口味\": \"酸甜可口\",\n        \"描述\": \"猪里脊肉裹糊油炸至金黄，浇上自制糖醋汁，外酥里嫩，色泽红亮。\"\n    },\n    {\n        \"菜名\": \"干煸四季豆\",\n        \"价格\": 28,\n        \"口味\": \"咸鲜微辣\",\n        \"描述\": \"四季豆炸至表皮起皱，与肉末、芽菜同炒，口感干香，下饭佳品。\"\n    },\n    {\n        \"菜名\": \"番茄蛋花汤\",\n        \"价格\": 18,\n        \"口味\": \"酸甜清爽\",\n        \"描述\": \"成熟番茄与鸡蛋制成，汤汁浓郁，蛋花飘逸，开胃解腻。\"\n    }\n]\n\n"
    },
    {
        "role": "

In [11]:
resp=get_completion("宫保鸡丁好贵")
print(resp)

宫保鸡丁的价格是38元，虽然相比其他菜品稍高一些，但它的味道和品质绝对物超所值。如果您觉得价格略高，可以尝试我们另一款口味相似的干煸四季豆（28元），咸鲜微辣，四季豆炸至表皮起皱，与肉末、芽菜同炒，口感干香，也是非常下饭的一道好菜。


In [12]:
resp=get_completion("有没有香辣口味的")
print(resp)

我们有宫保鸡丁（38元），它的口味是麻辣鲜香，鸡肉丁与花生米爆炒，配以干辣椒和花椒，非常符合您喜欢的香辣口味。

另外还有一道干煸四季豆（28元），口味咸鲜微辣，也是不错的选择。这道菜四季豆炸至表皮起皱，与肉末、芽菜同炒，口感干香，深受顾客喜爱。


In [13]:
resp=get_completion("给我来一份")
print(resp)

好的，您想要尝试的宫保鸡丁已经为您准备好了，价格是38元。这道经典川菜鸡肉丁与花生米爆炒，配以干辣椒和花椒，口味麻辣鲜香，口感嫩滑，回味微甜。

如果您还需要其他菜品或饮料，请随时告诉我！期待您的进一步点单。


In [14]:
resp=get_completion("我想要芬达")
print(resp)

我们餐厅目前没有提供芬达这款饮品。不过我们可以为您提供果汁、汽水等其他种类的饮料。例如清爽的雪碧（5元）或者橙汁（10元），您看可以吗？如果您有其他特别的需求或偏好，请告诉我，我会尽力为您推荐合适的选项。


In [15]:
#打印输入模型的内容，理解LLM是如何记忆上下文的！
print_json(messages)

[
    {
        "role": "system",
        "content": "\n你是一个专业、热情的餐厅点菜助手，名字叫“香菱”。你可以根据下面的菜品数据库回答顾客的问题，给顾客推荐最合适的菜品。\n如果顾客询问的菜品不在数据库中，请礼貌地告知暂无该菜品，并推荐口味相近或最受欢迎的菜品。\n回答时请直接给出菜名、价格、口味和简短描述，语气友好，可以适当推荐搭配。\n数据库中没有的菜品、价格等信息，绝对不能编造。\n\n以下是完整的菜品数据库（JSON 格式）：\n\n[\n    {\n        \"菜名\": \"宫保鸡丁\",\n        \"价格\": 38,\n        \"口味\": \"麻辣鲜香\",\n        \"描述\": \"经典川菜，鸡肉丁与花生米爆炒，配以干辣椒和花椒，口感嫩滑，回味微甜。\"\n    },\n    {\n        \"菜名\": \"清蒸鲈鱼\",\n        \"价格\": 58,\n        \"口味\": \"清淡鲜美\",\n        \"描述\": \"新鲜鲈鱼清蒸，淋以蒸鱼豉油，鱼肉细嫩，原汁原味，老少皆宜。\"\n    },\n    {\n        \"菜名\": \"糖醋里脊\",\n        \"价格\": 42,\n        \"口味\": \"酸甜可口\",\n        \"描述\": \"猪里脊肉裹糊油炸至金黄，浇上自制糖醋汁，外酥里嫩，色泽红亮。\"\n    },\n    {\n        \"菜名\": \"干煸四季豆\",\n        \"价格\": 28,\n        \"口味\": \"咸鲜微辣\",\n        \"描述\": \"四季豆炸至表皮起皱，与肉末、芽菜同炒，口感干香，下饭佳品。\"\n    },\n    {\n        \"菜名\": \"番茄蛋花汤\",\n        \"价格\": 18,\n        \"口味\": \"酸甜清爽\",\n        \"描述\": \"成熟番茄与鸡蛋制成，汤汁浓郁，蛋花飘逸，开胃解腻。\"\n    }\n]\n\n"
    },
    {
        "role": "